In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_47_try_1.pickle

In [ ]:
%%cudf.pandas.profile
### cell 48 ###

# Restore the variables into this cell's scope
question_of_interest_cell60 = 'Does your current employer incorporate machine learning methods into their business?'
title_for_x_axis_cell60 = ''

# Optimized for GPU with cudf.pandas extension
def combine_percentages_by_year(question, x_axis_title, include_2017=False):
    # Map years to their corresponding DataFrames
    years = ['2018', '2019', '2020', '2021', '2022']
    dfs   = [responses_df_2018_cell10,
             responses_df_2019_cell10,
             responses_df_2020,
             responses_df_2021,
             responses_df_2022_cell10]

    if include_2017:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)

    chunks = []
    for year, df in zip(years, dfs):
        # value_counts, division, multiplication and round all run on GPU
        counts     = df[question].value_counts(dropna=False).sort_index()
        total      = df[question].count()
        percentages = (counts * 100 / total).round(1)

        # reset_index and concat are GPU-accelerated
        tmp = percentages.reset_index()
        tmp.columns = [x_axis_title, 'percentage']
        tmp['year'] = year
        chunks.append(tmp)

    # Single concat call
    return pd.concat(chunks, ignore_index=True)

# Build the combined DataFrame and sort
maturity_df_combined = (
    combine_percentages_by_year(
        question_of_interest_cell60,
        title_for_x_axis_cell60
    )
    .sort_values(by=['year', 'percentage'], ascending=True)
)

maturity_df_combined.info()